In [1]:
"""
==============================================================================
🚀 [Step 4] L-channel CLAHE Preprocessing (Feature Enhancement)
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 저대비(Low Contrast) 환경에서 수집된 알약 각인의 식별력 향상을 위한
###                 적응형 히스토그램 평활화(CLAHE) 적용.
### @Logic        : RGB 왜곡 방지를 위해 LAB 색공간 분리 후 L(밝기) 채널만 선명도 보정.
### @Parameter    : clipLimit=2.0, tileGridSize=(8, 8)
==============================================================================
"""

'\n==============================================================================\n🚀 [Step 4] L-channel CLAHE Preprocessing (Feature Enhancement)\n==============================================================================\n### @Author       : 한의정 (Data Engineering Lead)\n### @Description  : 저대비(Low Contrast) 환경에서 수집된 알약 각인의 식별력 향상을 위한\n###                 적응형 히스토그램 평활화(CLAHE) 적용.\n### @Logic        : RGB 왜곡 방지를 위해 LAB 색공간 분리 후 L(밝기) 채널만 선명도 보정.\n### @Parameter    : clipLimit=2.0, tileGridSize=(8, 8)\n==============================================================================\n'

In [2]:
# ============================================================
# [Cell 0] 환경 설정 — Colab / 로컬 자동 감지
# ============================================================
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    sys.path.insert(0, os.path.abspath('..'))
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../data'))

print(f"✅ 환경: {'Colab' if is_colab else '로컬'}")
print(f"✅ PROJECT: {sys.path[0]}")
print(f"✅ DATA:    {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경: Colab
✅ PROJECT: /content/pill_detection_project
✅ DATA:    /content/drive/MyDrive/data/초급_프로젝트/dataset


In [3]:
############################################################
# 1. [Data Pipeline Phase 5] 
#     CLAHE 시력 교정 (알약 각인/음각 대비 극대화)
############################################################
import cv2
import numpy as np
import os
import glob
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

import os
import platform

# ==========================================
# 1. 환경 및 파라미터 설정
# ==========================================
train_dir = os.path.join(BASE_DIR, 'letterbox_images/train')  # ✅ NB03 출력물
val_dir   = os.path.join(BASE_DIR, 'letterbox_images/val')    # ✅ NB03 출력물

# 💡 [엔지니어링 포인트] CLAHE 엔진 초기화
# - clipLimit (2.0): 대비를 제한하는 임계값. 너무 높으면 이미지의 먼지(Noise)까지 증폭됩니다.
# - tileGridSize (8, 8): 이미지를 8x8 격자로 쪼개서 '국소적'으로 대비를 조정합니다.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe_to_folder(folder_path):
    """
    지정된 폴더의 이미지를 LAB 색공간으로 변환 후, 
    밝기(L) 채널의 대비만 극대화하여 덮어씁니다.
    """
    if not os.path.exists(folder_path):
        print(f"🚨 [에러] 폴더 경로를 찾을 수 없습니다: {folder_path}")
        return

    img_list = glob.glob(os.path.join(folder_path, "*.jpg"))
    if not img_list:
        print(f"⚠️ [알림] {os.path.basename(folder_path)} 폴더에 처리할 이미지가 없습니다.")
        return

    print(f"🔬 [전처리] {os.path.basename(folder_path)} 폴더 CLAHE 시력 교정 가동 (총 {len(img_list):,}장)")

    for img_path in tqdm(img_list, desc="CLAHE 적용 중"):
        # 1) 디코딩 (한글 경로 및 네트워크 드라이브 I/O 에러 원천 차단)
        img_array = np.fromfile(img_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        if img is None: continue

        # 2) BGR -> LAB 색공간 변환 (가장 중요한 핵심 로직)
        # RGB 채널 전체에 평활화를 걸면 알약 고유의 색상(Hue)이 심하게 왜곡됩니다.
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        
        # L: 밝기(Lightness), A/B: 색상 정보
        l, a, b = cv2.split(lab)

        # 3) L(밝기) 채널에만 적응형 히스토그램 평활화 적용
        l_clahe = clahe.apply(l)

        # 4) 채널 병합 및 원래 색공간(BGR)으로 복귀
        lab_merged = cv2.merge((l_clahe, a, b))
        final_img = cv2.cvtColor(lab_merged, cv2.COLOR_LAB2BGR)

        # 5) 디스크 제자리 덮어쓰기 (In-place Update)
        result, encoded_img = cv2.imencode('.jpg', final_img)
        if result:
            with open(img_path, mode='w+b') as f:
                encoded_img.tofile(f)

# ==========================================
# 2. 파이프라인 실행
# ==========================================
print("🚀 [Phase 5] 데이터셋 미세 각인 대비(Contrast) 최적화 작업을 시작합니다.\n")

apply_clahe_to_folder(train_dir)
apply_clahe_to_folder(val_dir)

print("\n✨ [데이터 엔지니어링 완수] 모든 이미지의 알약 각인과 음각이 선명하게 복원되었습니다!")
print("   이제 모델이 흐릿한 글자도 놓치지 않고 잡아낼 수 있습니다.")

🚀 [Phase 5] 데이터셋 미세 각인 대비(Contrast) 최적화 작업을 시작합니다.

🔬 [전처리] train 폴더 CLAHE 시력 교정 가동 (총 1,792장)


CLAHE 적용 중:   0%|          | 0/1792 [00:00<?, ?it/s]

🔬 [전처리] val 폴더 CLAHE 시력 교정 가동 (총 139장)


CLAHE 적용 중:   0%|          | 0/139 [00:00<?, ?it/s]


✨ [데이터 엔지니어링 완수] 모든 이미지의 알약 각인과 음각이 선명하게 복원되었습니다!
   이제 모델이 흐릿한 글자도 놓치지 않고 잡아낼 수 있습니다.
